# From a study record to a preliminary causal-support attribute
**Ted Smith — University of Louisville | Continuing education teaching demonstration**

This notebook replays the supplied ten-paper pilot. It does not re-read publications or call an AI service. The original scientific extractions and judgments were AI-authored and have not been validated by experts.

**Learning task:** trace a source-linked judgment to a decision input, apply the ordered rules, and explain what the result does—and does not—mean for a plain-language summary.

Run from the `python_demo` folder in an existing Jupyter environment using Python 3.10 or later. No additional Python packages, account, or API key are needed for the code below. The command-line alternative is `python course_demo.py`. Keep the JSON files and `run_pilot.py` beside this notebook. Outputs included here are a fresh replay of the existing records, not fresh AI readings.

## Step 1. Check the environment and load the records
This step checks the Python version, confirms the data files are present, and loads the records. The replay uses only Python's standard library: no machine-learning packages are installed and no model is trained.

In [1]:
import sys
import json
import copy
from pathlib import Path

assert sys.version_info >= (3, 10), "Use Python 3.10 or later."
ROOT = Path.cwd()
required = ["papers.json", "framework_v1.json", "sources.json", "run_pilot.py", "boundaries.json"]
missing = [name for name in required if not (ROOT / name).is_file()]
if missing:
    raise FileNotFoundError("Open this notebook in python_demo. Missing: " + ", ".join(missing))

from run_pilot import assign_indicator, validate_record, execute
papers = json.loads((ROOT / "papers.json").read_text(encoding="utf-8"))
framework = json.loads((ROOT / "framework_v1.json").read_text(encoding="utf-8"))
sources = {s["source_id"]: s for s in json.loads((ROOT / "sources.json").read_text(encoding="utf-8"))}
by_id = {p["paper_id"]: p for p in papers}
print(f"{len(papers)} publications; {len({p['cohort_family'] for p in papers})} underlying study families")
print("Framework version:", framework["version"])
print("No publication retrieval or AI call was performed.")

10 publications; 9 underlying study families
Framework version: 1.0.0
No publication retrieval or AI call was performed.


## Step 2. Inspect the exact question and its evidence
The acrolein example concerns minor eye irritation during a short exposure in healthy adults. Its label must not be transferred to chronic disease or another population. All six stored evidence items are shown, because the Hill rationales in Step 3 point to them by ID. Each is a stored, source-linked paraphrase from the pilot—not a newly checked quotation.

In [2]:
case = by_id["S02"]
print(case["short_citation"])
print(case["index_question"])
print("\nStored evidence:")
for evidence in case["evidence"]:
    print(evidence["evidence_id"], "|", evidence["locator"])
    print(evidence["text"])
    print()
print("Source URL:", sources["S02"]["source_read_url"])

Dwivedi et al., 2015
Does 2-hour acrolein exposure at 0.1 ppm produce minor acute eye irritation in healthy adults compared with clean air?

Stored evidence:
S02-E01 | Methods, pp. 811–813; Table 1
Eighteen healthy nonsmokers completed six 2-hour sessions, at least one week apart: clean air, 0.05 and 0.1 ppm acrolein, with and without an ethyl-acetate odor mask. Sequence described as balanced; random allocation not explicitly established.

S02-E02 | Table 1, p. 812
Chamber concentrations were measured, rather than relying only on nominal settings.

S02-E03 | Results, p. 814; Figure 2
At 118 minutes, median eye-irritation ratings were 0, 1.5 and 8 mm on a 100-mm scale at 0, 0.05 and 0.1 ppm; the overall test had P<0.001.

S02-E04 | Results; symptom time course and odor masking
The odor-mask condition did not account for the acrolein eye-irritation pattern, and ratings declined after exposure stopped.

S02-E05 | Results; lung function and inflammation
Other tested lung-function and infla

## Step 3. View all nine Bradford Hill fields
The rule engine does not add these statuses into a nine-item score. The AI authored the rationales and translated them into decision inputs. That translation is itself a scientific judgment requiring later human evaluation.

In [3]:
for domain in framework["hill_domains"]:
    entry = case["hill"][domain]
    print(domain.replace("_", " ").title(), "|", entry["status"])
    print(entry["rationale"])
    print("Evidence:", ", ".join(entry["evidence_ids"]))
    print()

Strength | supportive
The measured effect is small but experimentally separated from clean air; small magnitude does not automatically imply weak attribution.
Evidence: S02-E03

Consistency | external_unverified
Independent external evidence was not assessed. Background citations do not establish this domain in the present study-level pilot.
Evidence: S02-E06

Specificity | ambiguous
The eye outcome showed a pattern not shared by all tested outcomes; this is not proof of exclusive biological specificity.
Evidence: S02-E03, S02-E05

Temporality | supportive
Assigned exposure occurred before and during symptom measurements, with post-exposure follow-up.
Evidence: S02-E01, S02-E04

Biological Gradient | supportive
Eye-irritation ratings increased across the tested concentrations.
Evidence: S02-E03

Plausibility | external_unverified
The observed irritation is directly demonstrated; a molecular mechanism was not separately established in this pilot.
Evidence: S02-E03, S02-E06

Coherence | 

## Step 4. Inspect the inputs and apply the original rule
All values below were supplied by the AI assessment. A valid Boolean is not a verified fact. The expected original output is `('Stronger', 'R4')` for this narrow question.

In [4]:
for field, value in case["inputs"].items():
    print(f"{field}: {value}")
label, rule = assign_indicator(case["inputs"])
print("\nOutput:", label, rule)
print("Scope:", case["scope_statement"])

human_health_analysis: True
air_route_supported: True
claim_matches_measured_scope: True
essential_information_complete: True
temporality: supported
critical_alternative_explanation: False
exposure_contrast: controlled
controls_adequate_for_narrow_claim: True
robustness_support: True
precision_sufficient_for_claim: True

Output: Stronger R4
Scope: Stronger support refers only to minor acute eye irritation under these conditions—not chronic respiratory or cardiovascular disease.


## Step 5. Predict, then test a hypothetical change
**Before running:** What should happen when timing is changed from `supported` to `uncertain`?

The edit below is a synthetic exercise on a copy. It does not claim the acrolein paper actually had uncertain timing, and it does not re-run AI extraction.

In [5]:
original = copy.deepcopy(case["inputs"])
changed = copy.deepcopy(original)
changed["temporality"] = "uncertain"
print("Original:", assign_indicator(original))
print("Hypothetical:", assign_indicator(changed))
print("Original record preserved:", case["inputs"] == original)

Original: ('Stronger', 'R4')
Hypothetical: ('Limited', 'R2')
Original record preserved: True


## Step 6. Compare four teaching cases
The same ordinal category can describe different finding directions. S08 concerns the interpretability of **no enhancement detected**; it is not moderate support for harm or safety. The paper and question identifiers keep that distinction visible.

In [6]:
for paper_id in ["S03", "S02", "S08", "S06"]:
    p = by_id[paper_id]
    result = assign_indicator(p["inputs"])
    print(paper_id, p["short_citation"], "|", result)
    print("Finding:", p["finding"]["direction"])
    print("Main limitation:", p["main_limitation"])
    print("Scope:", p["scope_statement"])
    print()

S03 Rumchev et al., 2004 | ('Limited', 'R2')
Finding: Higher benzene associated with asthma case status
Main limitation: The first air measurement followed the asthma-related emergency presentation.
Scope: A positive association with case status is not proof that benzene caused a child to develop asthma.

S02 Dwivedi et al., 2015 | ('Stronger', 'R4')
Finding: Minor acute eye irritation increased
Main limitation: Small healthy-adult sample, subjective symptoms and brief exposure; sequence randomization not confirmed.
Scope: Stronger support refers only to minor acute eye irritation under these conditions—not chronic respiratory or cardiovascular disease.

S08 Ezratty et al., 2007 | ('Moderate', 'R5')
Finding: No enhancement detected; imprecise null-harm finding
Main limitation: Small sample, challenge-dose ceiling and no prespecified equivalence bound.
Scope: Moderate describes interpretability of a no-enhancement-detected result, not proof of safety or a harmful effect.

S06 Bard et al

## Step 7. Replay all ten records and the original 100 checks
The program writes the indicators and test outputs to `notebook_results`. Record structure, deterministic replay, synthetic gates, and broken-pointer controls are software checks—not an expert reference standard.

In [7]:
summary = execute(ROOT, ROOT / "notebook_results")
assert summary["tests_passed"] == summary["tests_total"]
assert summary["indicators"] == {"Moderate": 2, "Stronger": 1, "Limited": 7}
assert summary["scientific_accuracy_estimated"] is False
assert summary["expert_review_performed"] is False
print("\nThe observed 100/100 result does not estimate scientific accuracy.")

{
  "publications": 10,
  "development": 2,
  "demonstration": 8,
  "distinct_study_families": 9,
  "hill_domain_records": 90,
  "evidence_records": 60,
  "boundary_cases": 2,
  "indicators": {
    "Moderate": 2,
    "Stronger": 1,
    "Limited": 7
  },
  "triggered_rules": {
    "R5": 2,
    "R4": 1,
    "R2": 6,
    "R3": 1
  },
  "tests_total": 100,
  "tests_passed": 100,
  "test_categories": {
    "Architecture invariant": {
      "passed": 10,
      "total": 10
    },
    "Boundary rule": {
      "passed": 2,
      "total": 2
    },
    "Deterministic replay": {
      "passed": 10,
      "total": 10
    },
    "Negative control": {
      "passed": 3,
      "total": 3
    },
    "Record structure": {
      "passed": 10,
      "total": 10
    },
    "Rule branch": {
      "passed": 5,
      "total": 5
    },
    "Synthetic gate control": {
      "passed": 60,
      "total": 60
    }
  },
  "scientific_accuracy_estimated": false,
  "expert_review_performed": false,
  "independent_mod

## Step 8. Inspect a deliberately invalid input
This demonstrates what the checker can catch. It cannot determine whether a valid-looking judgment is scientifically correct.

In [8]:
invalid = copy.deepcopy(case["inputs"])
invalid["air_route_supported"] = "yes"  # Intentionally wrong datatype, not a scientific finding.
try:
    assign_indicator(invalid)
except ValueError as error:
    print("Expected rejection:", error)

Expected rejection: air_route_supported must be a JSON boolean


## Step 9. Draft the reader-facing attribute
Use S06 (Bard) and the participant worksheet. Write 60–90 words covering the provisional indicator, what supports interpretation, the main limitation, the exact scope, and AI disclosure. Do not turn higher odds into an absolute probability or a traffic-mixture association into proof of a benzene-only effect.

The text below is a stored source-based summary, not a new model output. The stored summary is a reference example of the content to cover. At about 180 words it is longer than your 60–90 word target, so aim to be more concise. Your classroom discussion is practice, not formal validation or expert review.

In [9]:
practice = by_id["S06"]
for paragraph in practice["plain_language_summary"]:
    print(paragraph, "\n")
print(practice["disclaimer"])

Researchers studied 2,134 heart attacks in Strasbourg. They compared each event day with other days for the same person, using modeled neighborhood air pollution. This approach helps control characteristics that do not change between the compared days. [S06, Methods] 

The pilot uses benzene on the preceding day, rather than selecting the largest association across time windows. Table 4 reports 7.2% higher odds per 1 microgram per cubic meter, with a 95% confidence interval from 0.3% to 14.5%. This corresponds to an odds ratio of 1.072. The exposure estimate precedes the event day. [S06, Table 4] 

The pilot nevertheless assigns limited support to a benzene-specific causal explanation. Benzene tracked other traffic pollutants, and the paper discusses particles that were not adequately measured. The design helps answer whether an exposure change preceded an event, but does not necessarily identify which component caused it. This distinction matters: a careful summary should not dismiss 

## Reflection
1. Which inputs required scientific judgment rather than coding?
2. Why does source-pointer validity not establish source meaning?
3. Why is this a study-specific support attribute rather than a body-of-evidence rating?
4. What expert and audience review would be needed before public-facing use?
5. Two groups of the 100 checks ("deterministic replay" and "title/background cannot bypass gates") cannot fail. Why not? What would a check that *could* fail look like?
6. Six of the ten publications are rated Limited by rule R2 because one field, `temporality`, is `uncertain`. Who should judge that field, and how would you test whether that judgment is reliable?

**Provenance:** Original pilot framework v1.0.0 and AI-authored records dated 25 September 2026. The framework, data, and original `run_pilot.py` were preserved. This notebook is a teaching adaptation. The original full-text publications and a verbatim AI-session log are not bundled.